# Excel Automation with Python

## Project Overview

This project uses Python and `openpyxl` to automate a simple Excel reporting workflow.

The notebook:
- Reads names and ages from an Excel worksheet.
- Filters records to keep people aged 25 or older.
- Sorts the filtered records from oldest to youngest.
- Creates a fresh `Automated Report` worksheet.
- Formats the report with a styled header, borders, column widths, an Excel table, and conditional formatting.
- Saves the updated workbook.

The report sheet is recreated each time the notebook runs, helping avoid duplicate report sheets.

## Tools
- Python
- openpyxl
- Jupyter Notebook
- Microsoft Excel

## Input and Output
The workbook file is `people_automation.xlsx`. The source data is read from its `Sheet` worksheet, using the first two columns (`Name` and `Age`). The generated report is saved back into the same workbook.

## Before Running
Install the dependency with `pip install openpyxl`, then place `people_automation.xlsx` in the same folder as this notebook. Run the cells from top to bottom.

## 1. Import the required libraries

In [ ]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.formatting.rule import CellIsRule

## 2. Load the workbook

In [ ]:
workbook = openpyxl.load_workbook("people_automation.xlsx")

## 3. Select the source worksheet

In [ ]:
sheet = workbook["Sheet"]

## 4. Read the source data

In [ ]:
people = []

for name, age in sheet.iter_rows(
    min_row=2,
    min_col=1,
    max_col=2,
    values_only=True,
):
    people.append([name, age])

## 5. Filter records

In [ ]:
# Keep people aged 25 or older.
filtered_people = [
    person for person in people
    if person[1] is not None and person[1] >= 25
]

## 6. Sort records from oldest to youngest

In [ ]:
sorted_people = sorted(
    filtered_people,
    key=lambda person: person[1],
    reverse=True,
)

## 7. Recreate the automated report worksheet

In [ ]:
# Remove the previous report so rerunning the notebook does not create duplicates.
if "Automated Report" in workbook.sheetnames:
    workbook.remove(workbook["Automated Report"])

report = workbook.create_sheet("Automated Report")

## 8. Write the report data

In [ ]:
report.append(["Name", "Age"])

for person in sorted_people:
    report.append(person)

## 9. Format the report header

In [ ]:
header_fill = PatternFill(fill_type="solid", fgColor="FFFF00")

for cell in report[1]:
    cell.font = Font(bold=True)
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center")

## 10. Set column widths

In [ ]:
report.column_dimensions["A"].width = 25
report.column_dimensions["B"].width = 10

## 11. Add borders

In [ ]:
thin_side = Side(style="thin")
report_border = Border(
    left=thin_side,
    right=thin_side,
    top=thin_side,
    bottom=thin_side,
)

for row in report.iter_rows():
    for cell in row:
        cell.border = report_border

## 12. Create a formatted Excel table

In [ ]:
last_row = report.max_row

table = Table(
    displayName="AutomatedPeopleTable",
    ref=f"A1:B{last_row}",
)

table_style = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False,
)

table.tableStyleInfo = table_style
report.add_table(table)

## 13. Add conditional formatting

In [ ]:
green_fill = PatternFill(
    start_color="C6EFCE",
    end_color="C6EFCE",
    fill_type="solid",
)
red_fill = PatternFill(
    start_color="FFC7CE",
    end_color="FFC7CE",
    fill_type="solid",
)

age_25_or_more = CellIsRule(
    operator="greaterThanOrEqual",
    formula=["25"],
    fill=green_fill,
)
age_under_25 = CellIsRule(
    operator="lessThan",
    formula=["25"],
    fill=red_fill,
)

if last_row >= 2:
    age_range = f"B2:B{last_row}"
    report.conditional_formatting.add(age_range, age_25_or_more)
    report.conditional_formatting.add(age_range, age_under_25)

## 14. Save the updated workbook

In [ ]:
workbook.save("people_automation.xlsx")